In [18]:
import torch
from recsys_learning.basic.features import SequenceFeature
from recsys_learning.basic.layers import EmbeddingLayer
import numpy as np

In [5]:
hist_seq = torch.tensor([[1, 2, 3, 4], [2, 3, 7, 8]])
hist_seq.shape

torch.Size([2, 4])

In [6]:
pos_seq = hist_seq
neg_seq = hist_seq

x = {'seq': hist_seq, 'pos': pos_seq, 'neg': neg_seq}
x

{'seq': tensor([[1, 2, 3, 4],
         [2, 3, 7, 8]]),
 'pos': tensor([[1, 2, 3, 4],
         [2, 3, 7, 8]]),
 'neg': tensor([[1, 2, 3, 4],
         [2, 3, 7, 8]])}

In [10]:
seq = SequenceFeature('seq', vocab_size=17, embed_dim=7, pooling='concat')
pos = SequenceFeature('pos', vocab_size=17, embed_dim=7, pooling='concat', shared_with='seq')
neg = SequenceFeature('neg', vocab_size=17, embed_dim=7, pooling='concat', shared_with='seq')

features = [seq, pos, neg]

In [11]:
item_emb = EmbeddingLayer(features)
item_emb

EmbeddingLayer(
  (embed_dict): ModuleDict(
    (seq): Embedding(17, 7)
  )
  (input_mask): InputMask()
)

In [ ]:
embedding = item_emb(x, features)
seq_embed, pos_embed, neg_embed = embedding[:, 0], embedding[:, 1], embedding[:, 2]
print(embedding.shape)
print(seq_embed.shape)
print(pos_embed.shape) 
print(neg_embed.shape)

torch.Size([2, 3, 4, 7])
torch.Size([2, 4, 7])
torch.Size([2, 4, 7])
torch.Size([2, 4, 7])


In [16]:
seq_embed *= 7**0.5
seq_embed.shape

torch.Size([2, 4, 7])

In [17]:
seq_embed = seq_embed.squeeze() 
seq_embed.shape

torch.Size([2, 4, 7])

In [22]:
positions = np.tile(np.array(range(hist_seq.shape[1])), [hist_seq.shape[0], 1])
positions

array([[0, 1, 2, 3],
       [0, 1, 2, 3]])

In [29]:
timeline_mask = torch.BoolTensor(hist_seq == 0)
timeline_mask
timeline_mask = ~timeline_mask.unsqueeze(-1)
timeline_mask

tensor([[[True],
         [True],
         [True],
         [True]],

        [[True],
         [True],
         [True],
         [True]]])

In [30]:
attention_mask = ~torch.tril(torch.ones((seq_embed.shape[1], seq_embed.shape[1]), dtype=torch.bool))
attention_mask

tensor([[False,  True,  True,  True],
        [False, False,  True,  True],
        [False, False, False,  True],
        [False, False, False, False]])

In [33]:
pos_logits = (seq_embed * pos_embed).sum(dim=-1)
pos_logits

tensor([[5.2195e-07, 9.9316e-07, 3.4431e-07, 6.5317e-07],
        [9.9316e-07, 3.4431e-07, 4.6325e-07, 6.6831e-07]],
       grad_fn=<SumBackward1>)

User TOwer

In [35]:
seq_embed = item_emb(x, features[:1])[:, 0]
seq_embed.shape

torch.Size([2, 4, 7])

In [37]:
seq = x['seq']
seq

tensor([[1, 2, 3, 4],
        [2, 3, 7, 8]])

In [38]:
seq_lens = (seq != 0).sum(dim=1) - 1
seq_lens

tensor([3, 3])

In [39]:
seq_lens = seq_lens.clamp(min=0)
seq_lens

tensor([3, 3])

In [40]:
batch_idx = torch.arange(seq_embed.size(0), device=seq_embed.device)
batch_idx

tensor([0, 1])

In [ ]:
user_emb = seq_embed[batch_idx, seq_lens]
user_emb.shape

tensor([[-7.6629e-06, -1.6339e-04, -5.6560e-05,  8.8725e-05, -1.0329e-04,
         -1.2524e-04, -1.7068e-04],
        [ 2.4123e-04,  2.3891e-05, -9.8869e-05,  4.0553e-05,  7.4536e-05,
          1.2761e-04,  5.8738e-05]], grad_fn=<IndexBackward0>)